# **Clasificación de Noticias para el Inglés (DOKU)**

Autor: Haritz Gómez

In [ ]:
!pip install -U transformers datasets evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import torch
from tqdm import tqdm
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Z1

### Cargar Datos

In [ ]:
from datasets import load_dataset

train = load_dataset("SetFit/bbc-news", split="train")
test_ds = load_dataset("SetFit/bbc-news", split="test")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
print(train[0])

In [ ]:
# Dividir test_ds en dev y test: primera mitad dev, segunda mitad test
len_test_ds = len(test_ds)
mid = int(len_test_ds/2)
dev = test_ds.select(range(mid))
test = test_ds.select(range(mid, len_test_ds))


In [ ]:
print(len(dev))
print(len(test))

## TOKENIZACIÓN

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

#tokenizer = AutoTokenizer.from_pretrained("roberta-base")
# model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)

In [ ]:
def tokenize_data(examples):
  # Tokeniza el texto de examples y añade su etiqueta al diccionario
  tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True, return_tensors='pt')
  # tokenized_inputs['labels'] = examples['label']
  return tokenized_inputs

In [ ]:
train_tokenized = train.map(tokenize_data, batched=True)  # Aplica tokenize_data a cada valor del diccionario
dev_tokenized = dev.map(tokenize_data, batched=True)
test_tokenized = test.map(tokenize_data, batched=True)

In [ ]:
# Se añade padding para que todas las noticias tengan la misma longitud
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## ENTRENAMIENTO

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=5)

# Mapeo de label_id y label
unique_labels = sorted(train.unique('label'))
id2label = {label_id: train.filter(lambda x: x['label'] == label_id)['label_text'][0] for label_id in unique_labels}
label2id = {v: k for k, v in id2label.items()}

# Añadir el mapeo en la configuración del modelo
model.config.id2label = id2label
model.config.label2id = label2id

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir="bert-lmrd",
                                  eval_strategy="epoch",
                                  load_best_model_at_end=True,
                                  save_total_limit=1,
                                  save_strategy="epoch",
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=64,
                                  per_device_eval_batch_size=64,
                                  num_train_epochs=5,
                                  weight_decay=0.01,
                                  fp16=True)



In [ ]:
!pip install evaluate

In [ ]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    # Load metrics
    accuracy = evaluate.load("accuracy")
    precision = evaluate.load("precision")
    recall = evaluate.load("recall")
    f1 = evaluate.load("f1")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [ ]:
trainer = Trainer(model=model,
          args=training_args,
					train_dataset=train_tokenized,
					eval_dataset=dev_tokenized,
					tokenizer=tokenizer,
					data_collator=data_collator,
					compute_metrics=compute_metrics)

In [ ]:
trainer.train()

In [ ]:
preds = trainer.predict(test_tokenized.select(range(100)))

In [ ]:
from pprint import pprint
pprint(preds.metrics)

## Guardar Modelo

Guardar el modelo y el tokenizador en un directorio específico.

In [ ]:
output_dir = "./bert_model"
trainer.save_model(output_dir)

In [ ]:
drive_output_dir = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/MODELOS_GUARDADOS"
trainer.save_model(drive_output_dir)
print(f"Modelo guardado en: {drive_output_dir}")

Cargar el modelo y tokenizador guardados

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Cargar el tokenizador
loaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Cargar el modelo
loaded_model = AutoModelForSequenceClassification.from_pretrained(output_dir)

## Encontrar mejores hiperparametros con sweep

La manera en la que he fine-tuneado el modelo arriba es la forma común de entrenar un modelo. No obstante, si nuestro objetivo es encontrar los mejores hiperparametros y el mejor modelo, podemos utilizar la herramienta wandb sweeps para crear una configuración de hiperparametros e ir haciendo más de un entrenamiento mientras estos hiperparametros se van probando. Una vez que tenemos los mejores entrenamientos para cada modelo, podemos probar que modelo fine-tuneado obtiene mejores resultados en el dataset test y asi quedarnos con el mejor modelo.

In [ ]:
import wandb
import os


In [ ]:
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'eval_f1',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'min': 1e-5,
            'max': 4e-5
        },
        'per_device_train_batch_size': {
            'values': [8, 16, 32]
        },
        'num_train_epochs': {
            'values': [3, 4, 5]
        },
        'weight_decay': {
            'min': 0.01,
            'max': 0.1
        },
        'warmup_ratio': {
            'min': 0.06,
            'max': 0.15
        }
    }
}


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

def train_with_sweep():
    # Inicializa una nueva corrida de Weights & Biases para este experimento
    run = wandb.init()

    # Accede a los hiperparámetros definidos por el sweep
    config = run.config

    # Re-inicializar el modelo que se va a entrenar
    # El tokenizador esta inicializado en las fases anteriores, importante que el tokenizador y el modelo sean del mismo modelo.
    model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

    unique_labels = sorted(train.unique('label'))
    id2label = {label_id: train.filter(lambda x: x['label'] == label_id)['label_text'][0] for label_id in unique_labels}
    label2id = {v: k for k, v in id2label.items()}

    model.config.id2label = id2label
    model.config.label2id = label2id

    # argumentos de entrenamiento utilizando la configuración del sweep
    training_args = TrainingArguments(
        output_dir=f"./roberta-doku-sweep-run-{run.id}", # Directorio de salida único para cada corrida, cambiar el nombre para cada modelo
        eval_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=1,
        save_strategy="epoch",
        learning_rate=config.learning_rate,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=64,
        num_train_epochs=config.num_train_epochs,
        weight_decay=config.weight_decay,
        fp16=True,
        report_to="wandb", # Reportar métricas a wandb
    )

    # Inicializa el Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=dev_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Entrena el modelo
    trainer.train()

    # Evalúa el modelo en el conjunto de desarrollo y guarda los resultados
    metrics = trainer.evaluate()
    wandb.log(metrics)

    # Guardar el mejor modelo y tokenizador en Google Drive
    drive_output_dir = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/MEJOR_MODELO_ROBERTA"     # Hay que cambiar el nombre para cada modelo
    trainer.save_model(drive_output_dir)
    tokenizer.save_pretrained(drive_output_dir)
    print(f"Modelo y tokenizador guardados en: {drive_output_dir}")

    # Finaliza la corrida de W&B
    wandb.finish()

In [ ]:
sweep_id = wandb.sweep(sweep_config, project="roberta-doku")

print(f"Sweep ID: {sweep_id}")

In [ ]:
# He probado los modelos haciendo 5 iteraciones ya que la variabilidad de los resultados es muy baja
wandb.agent(sweep_id, function=train_with_sweep, count=5)

Una vez que encontradas las mejores runs, voy a cargar los mejores modelos y a probar cual obtiene los mejores resultados con el dataset de test.

Me quedare con el modelo que mejores resultados obtenga.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


bert_model_path = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/MEJOR_MODELO_BERT"
distilbert_model_path = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/MEJOR_MODELO_DISTILBERT"
roberta_model_path = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/MEJOR_MODELO_ROBERTA"

# Load BERT tokenizer and model
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_path, local_files_only=True)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_path, local_files_only=True)
print(f"BERT Tokenizer loaded from: {bert_model_path}")
print(f"BERT Model loaded from: {bert_model_path}")

# Load DistilBERT tokenizer and model
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_model_path, local_files_only=True)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(distilbert_model_path, local_files_only=True)
print(f"DistilBERT Tokenizer loaded from: {distilbert_model_path}")
print(f"DistilBERT Model loaded from: {distilbert_model_path}")

# Load RoBERTa tokenizer and model
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_path, local_files_only=True)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_path, local_files_only=True)
print(f"RoBERTa Tokenizer loaded from: {roberta_model_path}")
print(f"RoBERTa Model loaded from: {roberta_model_path}")

In [ ]:
def tokenize_test_with(tokenizer):
    return lambda batch: tokenizer(
        batch["text"],
        truncation=True
    )


In [ ]:
test_tokenized_por_bert = test.map(
    tokenize_test_with(bert_tokenizer),
    batched=True,
    remove_columns=['text', 'label_text']
)

test_tokenized_por_distilbert = test.map(
    tokenize_test_with(distilbert_tokenizer),
    batched=True,
    remove_columns=['text', 'label_text']
)

test_tokenized_por_roberta = test.map(
    tokenize_test_with(roberta_tokenizer),
    batched=True,
    remove_columns=['text', 'label_text']
)

In [ ]:
from pprint import pprint
eval_args = TrainingArguments(
    output_dir="eval_output",
    per_device_eval_batch_size=64,
    do_train=False,
    do_eval=True,
    logging_strategy="no",
)

print("\n--- Evaluando modelo BERT ---")
bert_trainer = Trainer(
    model=bert_model,
    args=eval_args,
    eval_dataset=test_tokenized_por_bert,
    tokenizer=bert_tokenizer,
    compute_metrics=compute_metrics
)
bert_predictions = bert_trainer.predict(test_tokenized_por_bert)
print("Metricas del modelo BERT:")
pprint(bert_predictions.metrics)

print("\n--- Evaluando modelo DistilBERT ---")
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=eval_args,
    eval_dataset=test_tokenized_por_distilbert,
    tokenizer=distilbert_tokenizer,
    compute_metrics=compute_metrics
)
distilbert_predictions = distilbert_trainer.predict(test_tokenized_por_distilbert)
print("Metricas del modelo DistilBERT:")
pprint(distilbert_predictions.metrics)

print("\n--- Evaluando modelo RoBERTa ---")
roberta_trainer = Trainer(
    model=roberta_model,
    args=eval_args,
    eval_dataset=test_tokenized_por_roberta,
    tokenizer=roberta_tokenizer,
    compute_metrics=compute_metrics
)
roberta_predictions = roberta_trainer.predict(test_tokenized_por_roberta)
print("Metricas del modelo RoBERTa:")
pprint(roberta_predictions.metrics)

# Z2

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import bitsandbytes
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer


nf4_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=nf4_config,
    device_map="auto")


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Los datasets train y test se han cargado en Z1

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

# pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, dtype=torch.bfloat16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Dineño del prompt zero_shot y multiple_choice:

Primero craré un prompt sencillo en el que solo le indicaré su función al modelo, sin ejemplos (zero shot).

In [ ]:
labels = ['tech', 'business', 'sport', 'entertainment', 'politics']

In [ ]:
def extract_label(txt):
  # Find the position of 'assistant\n'
  assistant_delimiter = 'assistant\n'
  start_index = txt.find(assistant_delimiter)

  if start_index != -1:
    # Extract the text after 'assistant\n'
    extracted_text = txt[start_index + len(assistant_delimiter):]
  else:
    extracted_text = txt

  cleaned_label = extracted_text.strip().lower()

  # Check if the cleaned label is in the predefined labels list
  for label in labels:
    if label == cleaned_label:
      return label
  return 'unknown'

In [ ]:
def clasiffy_article_zero_shot(article_text):
  messages = [
  {
  "role": "system",
  "content": "You are an expert news classifier. Classify any given text into one of the following categories: tech, business, sport, entertainment, politics. Always answer with only the category name and always in lowercase."
  }
  ,
  {
  "role": "user",
  "content": f"Text: {article_text}\n\nWhat is the category?"
  }
  ]

  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  output_ids = model.generate(
      **inputs,
      max_new_tokens=10,
      eos_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  label = extract_label(decoded)   # Solo saco el label para poder compararlo con la etiqueta original y calcular las métricas
  return label

Ahora cambiaré el prompt para que tenga forma de multiple_choice, sin ejemplos:

In [ ]:
def classify_article_multiple_choice(article_text):
  messages = [
  {
  "role": "system",
  "content": "You are an expert news classifier. Classify any given text into one of the following categories: tech, business, sport, entertainment, politics. Always answer with only the category name and always in lowercase."
  }
  ,
  {
  "role": "user",
  "content": f"""Giving this text: {article_text}

Choose the correct category for the following news.

A) business
B) entertainment
C) politics
D) sport
E) tech

  Answer with only the category name:
  """
  }
  ]

  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  output_ids = model.generate(
      **inputs,
      max_new_tokens=10,
      eos_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  label = extract_label(decoded)
  return label

## Few Shots

Ahora probaré a añadir ejemplos en el prompt y comparar los resultados de zero_shot/multiple_choice con los resultados obtenidos al añadir uno, dos, tres y cinco ejemplos.

In [ ]:
def classify_article_few_shot(n_examples, article_text):
  examples = train.shuffle(seed=42).select(range(n_examples))

  # Build the examples string dynamically
  examples_str = ""
  for example in examples:
    examples_str += f"Text: {example['text']}\nCategory: {example['label_text']}\n\n"

  messages = [
  {
  "role": "system",
  "content": "You are an expert news classifier. Classify any given text into one of the following categories: tech, business, sport, entertainment, politics. Always answer with only the category name and always in lowercase."
  }
  ,
  {
  "role": "user",
  "content": f"""{examples_str}

      Text: {article_text}\nCategory (answer with only the category name):"""
  }
  ]

  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  output_ids = model.generate(
      **inputs,
      max_new_tokens=10,
      eos_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  label = extract_label(decoded)
  return label

## Calcular y Comparar Resultados

Utilizaré el dataset test de BBC para comparar los resultados de los distintos estilos de prompts.

In [ ]:
from sklearn.metrics import classification_report

def calculate_results_cero_and_multiple(test_dataset):
  # Esta función calcula los resultados de zero-shot y multiple-choice
  zero_shot_predicts = []
  zero_shot_labels = []
  multiple_choice_predicts = []
  multiple_choice_labels = []

  # Calcular resultados de 0-shot
  for i in range(len(test_dataset)):
    article = test_dataset[i]
    predicted_label = clasiffy_article_zero_shot(article['text'])
    true_label = article['label_text']
    zero_shot_predicts.append(predicted_label)
    zero_shot_labels.append(true_label)
  res_zero_shot = classification_report(zero_shot_labels, zero_shot_predicts, output_dict=True, zero_division=0)

  # Calcular resultados de multiple-choice
  for i in range(len(test_dataset)):
    article = test_dataset[i]
    predicted_label = classify_article_multiple_choice(article['text'])
    true_label = article['label_text']
    multiple_choice_predicts.append(predicted_label)
    multiple_choice_labels.append(true_label)
  res_multiple_choice = classification_report(multiple_choice_labels, multiple_choice_predicts, output_dict=True, zero_division=0)

  return [res_zero_shot, res_multiple_choice]

In [ ]:
from sklearn.metrics import classification_report

def calculate_few_shot_results(test_dataset):
  # Esta funcion calcula los resultados de los distintos few_shot.

    n_shots = [1, 3, 5]
    all_few_shot_metrics = {}

    for n in n_shots:
        predictions_list = []
        references_list = []

        for example in test_dataset:
            article_text = example['text']
            predicted_label = classify_article_few_shot(n, article_text)

            predictions_list.append(predicted_label)
            references_list.append(example['label_text'])

        # Generate a classification report for the current n_shot configuration
        report = classification_report(references_list, predictions_list, output_dict=True, zero_division=0)
        all_few_shot_metrics[f'{n}-shot'] = report

    return all_few_shot_metrics

Ahora visualizamos los datos

In [ ]:
from pprint import pprint
pprint("----------- Calculando resultados de zero_shot y multiple_choice:  ------------")
res_zero_shot, res_multiple_choice = calculate_results_cero_and_multiple(test)

pprint("----------- Resultados de Zero-Shot: ------------")
pprint(res_zero_shot)

pprint("----------- Resultados de Multiple-Choice: ------------")
pprint(res_multiple_choice)

('----------- Calculando resultados de zero_shot y multiple_choice: (UPDATED) '
 '------------')
'----------- Resultados de Zero-Shot: (UPDATED) ------------'
{'accuracy': 0.894,
 'business': {'f1-score': 0.8596491228070176,
              'precision': 0.8305084745762712,
              'recall': 0.8909090909090909,
              'support': 110.0},
 'entertainment': {'f1-score': 0.8874172185430463,
                   'precision': 0.8375,
                   'recall': 0.9436619718309859,
                   'support': 71.0},
 'macro avg': {'f1-score': 0.7468235382691923,
               'precision': 0.7540472002936743,
               'recall': 0.7444039673060859,
               'support': 500.0},
 'politics': {'f1-score': 0.8709677419354839,
              'precision': 0.9204545454545454,
              'recall': 0.826530612244898,
              'support': 98.0},
 'sport': {'f1-score': 0.9752066115702479,
           'precision': 0.959349593495935,
           'recall': 0.9915966386554622,
     

In [ ]:
from pprint import pprint
pprint("----------- Calculando resultados de few_shot:  ------------")
res_few_shot = calculate_few_shot_results(test)

pprint("----------- Resultados de Few-Shot: ------------")
pprint(res_few_shot)

'----------- Calculando resultados de few_shot:  ------------'
'----------- Resultados de Few-Shot: ------------'
{'1-shot': {'accuracy': 0.876,
            'business': {'f1-score': 0.8691588785046729,
                         'precision': 0.8942307692307693,
                         'recall': 0.8454545454545455,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8292682926829268,
                              'precision': 0.7311827956989247,
                              'recall': 0.9577464788732394,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7306158166232546,
                          'precision': 0.739389655779599,
                          'recall': 0.731239119634196,
                          'support': 500.0},
            'politics': {'f1-score': 0.8852459016393442,
                         'precision': 0.9529411764705882,
                         'recall': 0.826530612244898,
                       

Resultados de few-shots utilizando noticias enteras como ejemplos.

```
'----------- Calculando resultados de few_shot:  ------------'
'----------- Resultados de Few-Shot: ------------'
{'1-shot': {'accuracy': 0.876,
            'business': {'f1-score': 0.8691588785046729,
                         'precision': 0.8942307692307693,
                         'recall': 0.8454545454545455,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8292682926829268,
                              'precision': 0.7311827956989247,
                              'recall': 0.9577464788732394,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7306158166232546,
                          'precision': 0.739389655779599,
                          'recall': 0.731239119634196,
                          'support': 500.0},
            'politics': {'f1-score': 0.8852459016393442,
                         'precision': 0.9529411764705882,
                         'recall': 0.826530612244898,
                         'support': 98.0},
            'sport': {'f1-score': 0.9551020408163265,
                      'precision': 0.9285714285714286,
                      'recall': 0.9831932773109243,
                      'support': 119.0},
            'tech': {'f1-score': 0.8449197860962567,
                     'precision': 0.9294117647058824,
                     'recall': 0.7745098039215687,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.8821571696312371,
                             'precision': 0.8979351968082518,
                             'recall': 0.876,
                             'support': 500.0}},
 '3-shot': {'accuracy': 0.85,
            'business': {'f1-score': 0.8370044052863436,
                         'precision': 0.811965811965812,
                         'recall': 0.8636363636363636,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8496732026143791,
                              'precision': 0.7926829268292683,
                              'recall': 0.9154929577464789,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7208532128930978,
                          'precision': 0.74030837109113,
                          'recall': 0.7090532329022091,
                          'support': 500.0},
            'politics': {'f1-score': 0.8208092485549133,
                         'precision': 0.9466666666666667,
                         'recall': 0.7244897959183674,
                         'support': 98.0},
            'sport': {'f1-score': 0.9310344827586207,
                      'precision': 0.9557522123893806,
                      'recall': 0.907563025210084,
                      'support': 119.0},
            'tech': {'f1-score': 0.8865979381443299,
                     'precision': 0.9347826086956522,
                     'recall': 0.8431372549019608,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.8681253629289954,
                             'precision': 0.8949047996314871,
                             'recall': 0.85,
                             'support': 500.0}},
 '5-shot': {'accuracy': 0.85,
            'business': {'f1-score': 0.8465116279069768,
                         'precision': 0.8666666666666667,
                         'recall': 0.8272727272727273,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8322981366459627,
                              'precision': 0.7444444444444445,
                              'recall': 0.9436619718309859,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7225668373936417,
                          'precision': 0.7410834535834536,
                          'recall': 0.7118891431946232,
                          'support': 500.0},
            'politics': {'f1-score': 0.8571428571428571,
                         'precision': 0.9285714285714286,
                         'recall': 0.7959183673469388,
                         'support': 98.0},
            'sport': {'f1-score': 0.925764192139738,
                      'precision': 0.9636363636363636,
                      'recall': 0.8907563025210085,
                      'support': 119.0},
            'tech': {'f1-score': 0.8736842105263158,
                     'precision': 0.9431818181818182,
                     'recall': 0.8137254901960784,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.8709823502198877,
                             'precision': 0.9001323232323232,
                             'recall': 0.85,
                             'support': 500.0}}}

```



Como se puede observar, los resultados de zero-shot y multiple-choice son mejores que los resultados de todos los few-shot. Esto no es algo común y no debería pasar. Analizando los resultados, se puede ver que siempre infiere una etiqueta valida, aunque no sea correcta (nunca unknown). Asi que probablemente la perdida de rendimiento se deba a que los shots escogidos aleatoriamente desde el dataset de train son demasiado grandes y generan demasiado ruido al modelo.

Probare a utilizar ejemplos mas cortos y concretos para los ejemplos de los few-shots y ver si mejora el resultado respecto a utilizar noticias enteras como ejemplo.

In [ ]:
def classify_article_few_shot_sentences(n_examples, article_text):
  # Utiliza frases concretas para cada shot en vez de noticias enteras
  # En este caso también probaré con 1, 3 y 5-shots

  if n_examples == 1:
    examples = [
        {
             "text": "Oil prices rose sharply today as continued supply shortages pushed markets higher.",
             "label_text": "business"
        }
    ]


  elif n_examples == 3:
    examples = [
        {
            "text": "Oil prices rose sharply today as continued supply shortages pushed markets higher.",
            "label_text": "business"
        },
        {
            "text": "Major tech company unveiled a new smartphone featuring improved AI capabilities.",
            "label_text": "tech"
        },
        {
            "text": "The government announced a new economic stimulus package aimed at supporting households.",
            "label_text": "politics"
        }
    ]


  elif n_examples == 5:
    examples = [
        {
            "text": "Oil prices rose sharply today as continued supply shortages pushed markets higher.",
            "label_text": "business"
        },
        {
            "text": "Major tech company unveiled a new smartphone featuring improved AI capabilities.",
            "label_text": "tech"
        },
        {
            "text": "The government announced a new economic stimulus package aimed at supporting households.",
            "label_text": "politics"
        },
        {
            "text": "A new blockbuster film broke box office records during its opening weekend.",
            "label_text": "entertainment"
        },
        {
            "text": "The national football team secured a dramatic victory in the championship final.",
            "label_text": "sport"
        }
    ]


  examples_str = ""
  for example in examples:
    examples_str += f"Text: {example['text']}\nCategory: {example['label_text']}\n\n"

  messages = [
  {
  "role": "system",
  "content": "You are an expert news classifier. Classify any given text into one of the following categories: tech, business, sport, entertainment, politics. Always answer with only the category name and always in lowercase."
  }
  ,
  {
  "role": "user",
  "content": f"""{examples_str}

      Text: {article_text}\nCategory (answer with only the category name):"""
  }
  ]

  prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  output_ids = model.generate(
      **inputs,
      max_new_tokens=10,
      eos_token_id=tokenizer.eos_token_id
  )

  decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

  label = extract_label(decoded)
  return label

In [ ]:
from sklearn.metrics import classification_report

def calculate_few_shot_results_sentences(test_dataset):
  # Esta funcion calcula los resultados de los distintos few_shot.

    n_shots = [1, 3, 5]
    all_few_shot_metrics = {}

    for n in n_shots:
        predictions_list = []
        references_list = []

        for example in test_dataset:
            article_text = example['text']
            predicted_label = classify_article_few_shot_sentences(n, article_text)

            predictions_list.append(predicted_label)
            references_list.append(example['label_text'])
        # Generate a classification report for the current n_shot configuration
        report = classification_report(references_list, predictions_list, output_dict=True, zero_division=0)
        all_few_shot_metrics[f'{n}-shot'] = report

    return all_few_shot_metrics

In [ ]:
from pprint import pprint
pprint("----------- Calculando resultados de few_shot con frases como ejemplos:  ------------")
res_few_shot = calculate_few_shot_results_sentences(test)

pprint("----------- Resultados de Few-Shot con frases como shots: ------------")
pprint(res_few_shot)

('----------- Calculando resultados de few_shot con frases como ejemplos:  '
 '------------')
'----------- Resultados de Few-Shot con frases como shots: ------------'
{'1-shot': {'accuracy': 0.848,
            'business': {'f1-score': 0.825,
                         'precision': 0.7615384615384615,
                         'recall': 0.9,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8098159509202454,
                              'precision': 0.717391304347826,
                              'recall': 0.9295774647887324,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7048732646734185,
                          'precision': 0.7234344679923037,
                          'recall': 0.705356414866416,
                          'support': 500.0},
            'politics': {'f1-score': 0.8,
                         'precision': 0.9444444444444444,
                         'recall': 0.6938775510204082,
            

Resultados de few-shots utilizando ejemplos más cortos y concretos para cada etiqueta:


```
('----------- Calculando resultados de few_shot con frases como ejemplos:  '
 '------------')
'----------- Resultados de Few-Shot con frases como shots: ------------'
{'1-shot': {'accuracy': 0.848,
            'business': {'f1-score': 0.825,
                         'precision': 0.7615384615384615,
                         'recall': 0.9,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8098159509202454,
                              'precision': 0.717391304347826,
                              'recall': 0.9295774647887324,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7048732646734185,
                          'precision': 0.7234344679923037,
                          'recall': 0.705356414866416,
                          'support': 500.0},
            'politics': {'f1-score': 0.8,
                         'precision': 0.9444444444444444,
                         'recall': 0.6938775510204082,
                         'support': 98.0},
            'sport': {'f1-score': 0.9629629629629629,
                      'precision': 0.9435483870967742,
                      'recall': 0.9831932773109243,
                      'support': 119.0},
            'tech': {'f1-score': 0.8314606741573034,
                     'precision': 0.9736842105263158,
                     'recall': 0.7254901960784313,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.85209702774395,
                             'precision': 0.8777152329433646,
                             'recall': 0.848,
                             'support': 500.0}},
 '3-shot': {'accuracy': 0.852,
            'business': {'f1-score': 0.8533333333333334,
                         'precision': 0.8347826086956521,
                         'recall': 0.8727272727272727,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.7951807228915663,
                              'precision': 0.6947368421052632,
                              'recall': 0.9295774647887324,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.71420745435775,
                          'precision': 0.7317925879538777,
                          'recall': 0.7105148418735826,
                          'support': 500.0},
            'politics': {'f1-score': 0.8603351955307262,
                         'precision': 0.9506172839506173,
                         'recall': 0.7857142857142857,
                         'support': 98.0},
            'sport': {'f1-score': 0.9495798319327731,
                      'precision': 0.9495798319327731,
                      'recall': 0.9495798319327731,
                      'support': 119.0},
            'tech': {'f1-score': 0.8268156424581006,
                     'precision': 0.961038961038961,
                     'recall': 0.7254901960784313,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.8639450853694106,
                             'precision': 0.8906777411982598,
                             'recall': 0.852,
                             'support': 500.0}},
 '5-shot': {'accuracy': 0.894,
            'business': {'f1-score': 0.8699551569506726,
                         'precision': 0.8584070796460177,
                         'recall': 0.8818181818181818,
                         'support': 110.0},
            'entertainment': {'f1-score': 0.8472222222222222,
                              'precision': 0.8356164383561644,
                              'recall': 0.8591549295774648,
                              'support': 71.0},
            'macro avg': {'f1-score': 0.7474522348548197,
                          'precision': 0.7549429754512874,
                          'recall': 0.7411092307174684,
                          'support': 500.0},
            'politics': {'f1-score': 0.9015544041450777,
                         'precision': 0.9157894736842105,
                         'recall': 0.8877551020408163,
                         'support': 98.0},
            'sport': {'f1-score': 0.9747899159663865,
                      'precision': 0.9747899159663865,
                      'recall': 0.9747899159663865,
                      'support': 119.0},
            'tech': {'f1-score': 0.8911917098445595,
                     'precision': 0.945054945054945,
                     'recall': 0.8431372549019608,
                     'support': 102.0},
            'unknown': {'f1-score': 0.0,
                        'precision': 0.0,
                        'recall': 0.0,
                        'support': 0.0},
            'weighted avg': {'f1-score': 0.9022034621054289,
                             'precision': 0.9117930374020132,
                             'recall': 0.894,
                             'support': 500.0}}}
```



Comparando los resultados, se puede ver cierta mejora utilizando frases mas cortas y concretas en vez de noticias enteras como shots, llegando a un máximo  f1: 0.9022 utilizando frases vs f1: 0.87 utilizando noticias enteras (ambos casos con 5-shot).

También se puede ver como aumentando la cantidad de ejemplos se han conseguido mejores resultados:

 [1-shot: f1-score': 0.85209702774395, 3-shot: 'f1-score': 0.8639450853694106, 5-shot: f1-score': 0.9022034621054289]

# Z3

## Preparación de Datos

Probaré con datasets reducidos de 50 (10 por clase), 250 (50 por clase) y 500 (100 por clase) datos.

Mi objetivo es ver como afecta el aumento de datos a los distintos datasets iniciales y si merece la pena hacer un aumento de datos a un dataset no tan pequeño como el de 500 datos iniciales.

In [ ]:
from datasets import load_dataset

ds = load_dataset("SetFit/bbc-news")
train = ds['train']
dev = ds['test'].select(range(0,500))
test_final = ds['test'].select(range(500,1000))

# Mapeo de label_id y label
unique_labels = sorted(train.unique('label'))
id2label = {label_id: train.filter(lambda x: x['label'] == label_id)['label_text'][0] for label_id in unique_labels}
label2id = {v: k for k, v in id2label.items()}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/880 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1225 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/1225 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1225 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1225 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1225 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1225 [00:00<?, ? examples/s]

In [ ]:
import random
def stratified_sample(train_ds, n_per_class, seed=0):
  # Crea los datasets reducidos con la cantidad de ejemplos por clase indicada
    rng = random.Random(seed)
    selected_indices = []
    # Iterar sobre los IDs enteros únicos de las etiquetas
    for lbl_id in sorted(train_ds.unique('label')):
        idxs = [i for i, x in enumerate(train_ds) if x['label'] == lbl_id]
        rng.shuffle(idxs)
        selected = idxs[:n_per_class]
        selected_indices.extend(selected)
    return train_ds.select(selected_indices)

In [ ]:
n_per_class = [10, 50, 100]

dataset_50 = stratified_sample(train, n_per_class[0], seed=42)

dataset_250 = stratified_sample(train, n_per_class[1], seed=42)

dataset_500_inicial = stratified_sample(train, n_per_class[2], seed=42)


In [ ]:
print(dataset_500_inicial)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 500
})


Cargo el modelo de Z2 que se encargará de la generación de datos:

In [ ]:
import bitsandbytes
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer


nf4_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=nf4_config,
    device_map="auto")


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from transformers import pipeline, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

# pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, dtype=torch.bfloat16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def generate_augmented_examples(label_text_str, n_aug, seed=0):
    aug_examples = []

    integer_label = label2id[label_text_str]

    for i in range(n_aug):
        messages = [
            {"role": "system", "content": "You are a helpful news article writer. Generate short BBC-style news paragraphs."},
            {"role": "user", "content": f"Generate a short BBC-style news paragraph about the category '{label_text_str}'. Do not include the label word in the text. Keep length similar to dataset examples."}
        ]


        prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_formatted, return_tensors="pt").to(model.device)

        # Ensure the seed changes for each example
        torch.manual_seed(seed + i)


        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7 # Adjust temperature for creativity vs coherence
        )

        decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)


        # Extract only the generated response.
        assistant_tag = "assistant\n"
        start_index = decoded.rfind(assistant_tag)
        if start_index != -1:
            generated_text = decoded[start_index + len(assistant_tag):].strip()
        else:
            generated_text = decoded.strip()

        aug_examples.append({"text": generated_text, "label": integer_label})
    return aug_examples

Para aumentar los datasets, hare lo siguiente:


*   50 ejemplos ->  generar 150 ejemplos y añadir los 50 iniciales (200 ejemplos)
*   250 ejemplos -> generar 250 ejemplos y añadir los 250 iniciales (500 ejemplos)
*   500 ejemplos -> generar 500 ejemplos y añadir los 500 iniciales (1000 ejemplos)





In [ ]:
from datasets import Dataset
# Aumentar dataset de 50 a 200
dataset_200 = []
dataset_200.extend(dataset_50)
for label_name in id2label.values():
  dataset_200.extend(generate_augmented_examples(label_name, 30, seed=42)) # Genera 30 ejemplos por clase

dataset_200_hf = Dataset.from_list(dataset_200)
dataset_200_shuffled = dataset_200_hf.shuffle(seed=42)
drive_output_dir_200 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_200"
dataset_200_shuffled.save_to_disk(drive_output_dir_200)
print(f"Dataset 200 saved to: {drive_output_dir_200}")
print(dataset_200_shuffled)

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset 200 saved to: /content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_200
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 200
})


In [ ]:
print(dataset_200_shuffled[0])

{'text': "A popular actress is set to star in a new blockbuster movie, bringing her iconic character back to life on screen for audiences worldwide. The film promises an exciting and emotional journey, with stunning visuals and heartwarming moments that will leave viewers spellbound. Fans of the actress can't wait to see their favorite characters brought to life on the big screen once again.", 'label': 3, 'label_text': None}


In [ ]:
# Aumentar dataset de 250 a 500
dataset_500 = []
dataset_500.extend(dataset_250)
for label_name in id2label.values():
  dataset_500.extend(generate_augmented_examples(label_name, 50, seed=42)) # Genera 50 ejemplos por clase

dataset_500_hf = Dataset.from_list(dataset_500)
dataset_500_shuffled = dataset_500_hf.shuffle(seed=42)
drive_output_dir_500 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_500"
dataset_500_shuffled.save_to_disk(drive_output_dir_500)
print(f"Dataset 500 saved to: {drive_output_dir_500}")
print(dataset_500_shuffled)

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset 500 saved to: /content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_500
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 500
})


In [ ]:
print(dataset_500_shuffled[0])

{'text': 'The government announced plans to increase funding for public transport, aiming to improve accessibility and reduce traffic congestion. This comes as concerns over climate change continue to grow, prompting policymakers to explore innovative solutions to combat rising emissions. The move is part of a broader strategy to enhance urban infrastructure, with the goal of creating more sustainable living environments.', 'label': 4, 'label_text': None}


In [ ]:
# Aumentar dataset de 500 a 1000
dataset_1000 = []
dataset_1000.extend(dataset_500_inicial)
for label_name in id2label.values():
  dataset_1000.extend(generate_augmented_examples(label_name, 100, seed=42)) # Genera 100 ejemplos por clase

dataset_1000_hf = Dataset.from_list(dataset_1000)
dataset_1000_shuffled = dataset_1000_hf.shuffle(seed=42)
drive_output_dir_1000 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_1000"
dataset_1000_shuffled.save_to_disk(drive_output_dir_1000)
print(f"Dataset 1000 saved to: {drive_output_dir_1000}")
print(dataset_1000_shuffled)

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset 1000 saved to: /content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_1000
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 1000
})


In [ ]:
print(dataset_1000[897])

{'text': "In a surprise move, the popular actor will be appearing at this year's awards ceremony after years of absence. His fans have been eagerly anticipating his return and many believe it is an indication of his newfound success. The cast will be joined by other industry heavyweights as they celebrate their achievements with champagne and cake.", 'label': 3}


Cargar los datasets aumentados:

In [ ]:
from datasets import load_from_disk

drive_output_dir_200 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_200"
drive_output_dir_500 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_500"
drive_output_dir_1000 = "/content/drive/MyDrive/Colab Notebooks/PLN/PROYECTO/dataset_1000"

dataset_200_loaded = load_from_disk(drive_output_dir_200)
dataset_500_loaded = load_from_disk(drive_output_dir_500)
dataset_1000_loaded = load_from_disk(drive_output_dir_1000)

print("Dataset 200 Loaded:")
print(dataset_200_loaded)
print("\nDataset 500 Loaded:")
print(dataset_500_loaded)
print("\nDataset 1000 Loaded:")
print(dataset_1000_loaded)

Dataset 200 Loaded:
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 200
})

Dataset 500 Loaded:
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 500
})

Dataset 1000 Loaded:
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 1000
})


Creé los datasets aumentados sin label_text, asi que creare los label_text a partir de mapear label.

In [ ]:
def add_label_text(example):
  example['label_text'] = id2label[example['label']]
  return example

dataset_200_loaded = dataset_200_loaded.map(add_label_text)
dataset_500_loaded = dataset_500_loaded.map(add_label_text)
dataset_1000_loaded = dataset_1000_loaded.map(add_label_text)

print("Label_text added to dataset_200_loaded:")
print(dataset_200_loaded[0])
print("\nLabel_text added to dataset_500_loaded:")
print(dataset_500_loaded[0])
print("\nLabel_text added to dataset_1000_loaded:")
print(dataset_1000_loaded[0])

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Label_text added to dataset_200_loaded:
{'text': 'Shares rise as investors bet on economic recovery, with tech stocks leading gains following positive corporate earnings reports.', 'label': 1, 'label_text': 'business'}

Label_text added to dataset_500_loaded:
{'text': 'The government announced plans to increase funding for public transport, aiming to improve accessibility and reduce traffic congestion. This comes as concerns over climate change continue to grow, prompting policymakers to explore innovative solutions to combat rising emissions. The move is part of a broader strategy to enhance urban infrastructure, with the goal of creating more sustainable living environments.', 'label': 4, 'label_text': 'politics'}

Label_text added to dataset_1000_loaded:
{'text': 'The government announced plans for major infrastructure projects, including new roads and bridges across the country. This comes as concerns grow over the increasing cost of living and the impact on working-class families.

## Entrenamiento

Ahora que tengo los datasets listos, fine-tunearé un modelo DistilBERT con los datasets iniciales y los aumentados, utilizando los mismos hiperparámetros, después compararé los resultados para ver si ese aumento de datos ha merecido la pena para los distintos datasets iniciales.

Aunque en Z1 el modelo DistilBERT no ha sido el mejor para la tarea de clasificación, como en este apartado lo importante no es maximizar los resultados, utilizare DistilBERT por ser mas pequeño y rápido. Utilizaré los mejores hiperparámetros encontrados en Z1 para Distilbert.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_data(examples):
  # Tokeniza el texto de examples y añade su etiqueta al diccionario
  tokenized_inputs = tokenizer(examples['text'], truncation=True, padding=True, return_tensors='pt')
  return tokenized_inputs

In [ ]:
# Se añade padding para que todas las noticias tengan la misma longitud
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
!pip install evaluate       # A veces no se instala correctamente y hace falta para la funcion compute_metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    # Load metrics
    accuracy = evaluate.load("accuracy")
    precision = evaluate.load("precision")
    recall = evaluate.load("recall")
    f1 = evaluate.load("f1")

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }

In [ ]:
# Define all datasets to be used for training
train_datasets = {
    "dataset_50_original": dataset_50,
    "dataset_250_original": dataset_250,
    "dataset_500_original": dataset_500_inicial,
    "dataset_200_augmented": dataset_200_loaded,
    "dataset_500_augmented": dataset_500_loaded,
    "dataset_1000_augmented": dataset_1000_loaded,
}

# Tokenizar dev y test para la evaluación
dev_tokenized_z3 = dev.map(tokenize_data, batched=True)
test_final_tokenized_z3 = test_final.map(tokenize_data, batched=True)

print("Development set tokenized:")
print(dev_tokenized_z3)
print("\nFinal test set tokenized:")
print(test_final_tokenized_z3)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Development set tokenized:
Dataset({
    features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
    num_rows: 500
})

Final test set tokenized:
Dataset({
    features: ['text', 'label', 'label_text', 'input_ids', 'attention_mask'],
    num_rows: 500
})


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from pprint import pprint
all_results = {}

for name, train_ds in train_datasets.items():
    print(f"\n--- Training with {name} dataset ---")

    # Tokenizar el train actual
    current_train_tokenized = train_ds.map(tokenize_data, batched=True)

    # Re-inicializar modelo en cada run
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=5)
    model.config.id2label = id2label
    model.config.label2id = label2id

    training_args = TrainingArguments(
        # Hare todos los entrenamientos con los mejores hiperparametros de Z1 para Distilbert (5 epochs en vez de 4)
        output_dir=f"./distilbert-z3-{name}",
        eval_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=1,
        save_strategy="epoch",
        learning_rate=5.7e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        num_train_epochs=5,
        weight_decay=0.0070,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=current_train_tokenized,
        eval_dataset=dev_tokenized_z3,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # Evaluar el modelo con el dataset test
    preds = trainer.predict(test_final_tokenized_z3)
    all_results[name] = preds.metrics
    print(f"Metrics for {name}:")
    pprint(preds.metrics)


print("\n--- Summary of all training results ---")
for name, metrics in all_results.items():
    print(f"\n{name}:")
    pprint(metrics)


--- Training with dataset_50_original dataset ---


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-599229563.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.500830,0.608000,0.806753,0.608000,0.561167
2,No log,1.352891,0.668000,0.821056,0.668000,0.620456
3,No log,1.227241,0.822000,0.879440,0.822000,0.818391
4,No log,1.138809,0.852000,0.899918,0.852000,0.852192
5,No log,1.101694,0.860000,0.899015,0.860000,0.858008


Metrics for dataset_50_original:
{'test_accuracy': 0.87,
 'test_f1': 0.8650645215498626,
 'test_loss': 1.0975463390350342,
 'test_precision': 0.8930507974495902,
 'test_recall': 0.87,
 'test_runtime': 9.7423,
 'test_samples_per_second': 51.322,
 'test_steps_per_second': 0.821}

--- Training with dataset_250_original dataset ---


...

Metrics for dataset_500_augmented:
{'test_accuracy': 0.972,
 'test_f1': 0.9719270151519248,
 'test_loss': 0.11618699878454208,
 'test_precision': 0.9721501017188253,
 'test_recall': 0.972,
 'test_runtime': 9.8171,
 'test_samples_per_second': 50.932,
 'test_steps_per_second': 0.815}

--- Training with dataset_1000_augmented dataset ---


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-599229563.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.232556,0.936000,0.940811,0.936000,0.934051
2,No log,0.232517,0.946000,0.951166,0.946000,0.945538
3,No log,0.141289,0.966000,0.967111,0.966000,0.965828
4,No log,0.074879,0.980000,0.980280,0.980000,0.980069
5,No log,0.076210,0.978000,0.978220,0.978000,0.978020


Metrics for dataset_1000_augmented:
{'test_accuracy': 0.978,
 'test_f1': 0.9780140831185132,
 'test_loss': 0.10198550671339035,
 'test_precision': 0.9782385453529522,
 'test_recall': 0.978,
 'test_runtime': 9.7637,
 'test_samples_per_second': 51.21,
 'test_steps_per_second': 0.819}

--- Summary of all training results ---

dataset_50_original:
{'test_accuracy': 0.87,
 'test_f1': 0.8650645215498626,
 'test_loss': 1.0975463390350342,
 'test_precision': 0.8930507974495902,
 'test_recall': 0.87,
 'test_runtime': 9.7423,
 'test_samples_per_second': 51.322,
 'test_steps_per_second': 0.821}

dataset_250_original:
{'test_accuracy': 0.964,
 'test_f1': 0.9636309602730986,
 'test_loss': 0.1509338617324829,
 'test_precision': 0.9646184586456396,
 'test_recall': 0.964,
 'test_runtime': 10.0505,
 'test_samples_per_second': 49.749,
 'test_steps_per_second': 0.796}

dataset_500_original:
{'test_accuracy': 0.97,
 'test_f1': 0.9699870810607938,
 'test_loss': 0.11945533752441406,
 'test_precision': 0.970